# 06 — Quarto : publication scientifique reproductible

Ce notebook couvre l'installation et la prise en main de Quarto dans l'écosystème Python/AI.

## 1. Qu'est-ce que Quarto ?

Quarto est un système de publication scientifique et technique développé par Posit (anciennement RStudio). C'est le successeur de R Markdown, mais **language-agnostic** — il supporte Python, R, Julia et Observable.

**Ce que Quarto fait :**
- Prend un fichier `.qmd` (Quarto Markdown) ou un notebook `.ipynb` comme source
- Exécute le code Python/R/Julia via un moteur (Jupyter ou Knitr)
- Génère un rendu : HTML, PDF, Word, slides (Reveal.js), site web statique, livre

**Analogie C++ :** c'est comme un compilateur, mais pour des documents. Le `.qmd` est ton source, Quarto est le compiler, le HTML/PDF est le binaire.

### Architecture interne

```
[Quarto CLI]  →  [Jupyter kernel ai_learning]  →  exécute le code Python
      ↓
  [Pandoc intégré]
      ↓
  HTML / PDF / Slides
```

Pandoc est **bundlé dans Quarto** — pas besoin de l'installer séparément.

### Différence clé avec Jupyter

| | Jupyter | Quarto |
|---|---|---|
| Format source | `.ipynb` (JSON lourd) | `.qmd` (texte Markdown pur) |
| Versioning Git | Difficile (diff JSON + outputs) | Propre (texte diff-able) |
| Output | Interface web interactive | Documents publiables |
| Reproductibilité | Manuel | Automatique (re-execute at render) |

### Cas d'usage en Python/AI
- Transformer des notebooks en documentation propre
- Publier des analyses sur GitHub Pages
- Créer des rapports reproductibles (execute + render en une commande)
- Présenter des résultats de ML avec code, graphiques et texte intégrés

## 2. Installation

Quarto est un **binaire standalone** — il ne s'installe pas via conda ou pip.
Il s'installe comme un outil système, indépendamment de tes environnements conda,
et détecte automatiquement les kernels Jupyter enregistrés.

### Audit préalable

Avant d'installer, vérifier si Quarto est déjà présent :

In [ ]:
!quarto --version

### Installation via winget (Windows 11)

`winget` est le gestionnaire de paquets natif de Windows 11. Il télécharge l'installeur
officiel et l'exécute silencieusement.

À exécuter dans un terminal **PowerShell** (pas dans VSCode) :

```powershell
winget install --id Posit.Quarto
```

### Dépendances Python requises dans l'env conda

Quarto délègue l'exécution du code Python au kernel Jupyter, mais son moteur a besoin
de quelques packages Python dans l'environnement `ai_learning` pour orchestrer ce rendu.
Ces packages ne sont **pas** installés automatiquement par winget.

À exécuter dans PowerShell après l'installation de Quarto :

```powershell
conda activate ai_learning
conda install pyyaml nbformat nbclient ipywidgets
```

| Package | Rôle |
|---|---|
| `pyyaml` | Lire les métadonnées YAML des fichiers `.qmd` |
| `nbformat` | Lire/écrire le format `.ipynb` |
| `nbclient` | Exécuter les cellules d'un notebook programmatiquement |
| `ipywidgets` | Support des widgets interactifs dans les outputs |

### Problème connu : PATH non mis à jour après winget

Sur certaines configurations Windows, `winget install` installe le binaire dans
`C:\Program Files\Quarto\bin\` mais **n'ajoute pas ce chemin au PATH système**.

**Symptôme :** `quarto` fonctionne dans PowerShell mais pas dans le kernel Jupyter.

**Cause :** Le kernel Jupyter hérite du PATH au moment où il est lancé par VSCode.
Chaque processus hérite une **copie figée** du PATH de son parent au démarrage :

```
Registre Windows (HKCU\Environment)
        ↓  héritage au démarrage
    Processus VSCode
        ↓  héritage à la création du kernel
    Processus Python (kernel ai_learning)
```

Si `C:\Program Files\Quarto\bin` n'est pas dans le registre, il est absent de toute
la chaîne — même après restart du kernel ou de VSCode.

**Fix permanent — via PowerShell :**

```powershell
[System.Environment]::SetEnvironmentVariable("Path", [System.Environment]::GetEnvironmentVariable("Path", "User") + ";C:\Program Files\Quarto\bin", "User")
```

Cette commande écrit directement dans `HKCU\Environment` (registre utilisateur).
Vérifier que le chemin est bien enregistré :

```powershell
[System.Environment]::GetEnvironmentVariable("Path", "User")
```

Tu dois voir `C:\Program Files\Quarto\bin` dans la sortie. Ensuite **fermer et rouvrir VSCode**.

**Workaround temporaire** (session kernel courante uniquement) :

In [ ]:
import os
os.environ["PATH"] += r";C:\Program Files\Quarto\bin"

In [ ]:
import os
for p in os.environ.get("PATH", "").split(";"):
    print(p)

> Cette cellule modifie le PATH du processus kernel pour la session courante.
> Elle devient inutile une fois le PATH système corrigé via Windows Settings.

## 3. Vérification de l'installation

Vérifier la version installée :

In [ ]:
!quarto --version

Vérifier que Quarto détecte bien le kernel `ai_learning` :

In [ ]:
!quarto check jupyter

> `quarto check jupyter` inspecte l'environnement Jupyter disponible et liste les kernels détectés.
> Tu dois voir `ai_learning` parmi les kernels disponibles, et la ligne `Checking Jupyter engine render....OK` sans erreur.

## 4. Premier document Quarto

### Le format `.qmd`

Un fichier `.qmd` (Quarto Markdown) est un fichier texte pur composé de deux parties :

1. **YAML front matter** — métadonnées du document (entre `---`)
2. **Corps** — Markdown standard + blocs de code délimités par ` ``` `

```
---
title: "Mon analyse"
format: html
jupyter: ai_learning
---

## Introduction

Texte en Markdown standard.

```{python}
import numpy as np
print(np.version.version)
```
```

La clé `jupyter: ai_learning` indique à Quarto quel kernel utiliser pour exécuter le code.

### Commandes essentielles

| Commande | Effet |
|---|---|
| `quarto render document.qmd` | Exécute le code et génère le fichier de sortie |
| `quarto preview document.qmd` | Rendu live dans le navigateur, se met à jour à chaque sauvegarde |
| `quarto render document.qmd --to pdf` | Force la sortie en PDF (nécessite LaTeX) |
| `quarto render document.qmd --to docx` | Force la sortie en Word |

### Créer un premier document de test

In [ ]:
qmd_content = """\
---
title: "Test Quarto"
format: html
jupyter: ai_learning
---

## Test Python

```{python}
print("Hello from Quarto!")
import sys
print(f"Python {sys.version}")
```
"""

with open("test_quarto.qmd", "w", encoding="utf-8") as f:
    f.write(qmd_content)

print("Fichier test_quarto.qmd créé.")

Rendre le document en HTML :

In [ ]:
!quarto render test_quarto.qmd

> Quarto exécute le code Python via le kernel `ai_learning`, puis génère `test_quarto.html`
> dans le même dossier. Ouvre ce fichier dans un navigateur pour voir le résultat.

### Nettoyer les fichiers de test

In [ ]:
import os
for f in ["test_quarto.qmd", "test_quarto.html"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Supprimé : {f}")

## 5. Exemples pratiques

Les exemples suivants créent chacun un fichier `.qmd`, le rendent en HTML, puis le nettoient.
Chaque exemple introduit un concept Quarto spécifique.

---

### Exemple 1 — Contrôle de l'affichage du code

Par défaut, Quarto affiche le code **et** le résultat. Les **cell options** (commentaires `#|`)
permettent de contrôler ce comportement cellule par cellule.

| Option | Valeur | Effet |
|---|---|---|
| `echo` | `true` / `false` | Affiche ou cache le code source |
| `output` | `true` / `false` | Affiche ou cache le résultat |
| `warning` | `true` / `false` | Affiche ou cache les warnings Python |
| `include` | `true` / `false` | Inclut ou exclut entièrement la cellule |

Ces options s'écrivent en commentaires YAML dans la cellule — Quarto les intercepte,
Python les ignore (ce sont des commentaires valides `#`).

In [ ]:
qmd = """\
---
title: "Exemple 1 — Cell options"
format: html
jupyter: ai_learning
---

Cette cellule affiche le code ET le résultat (comportement par défaut) :

```{python}
print("Code visible + résultat visible")
```

Cette cellule cache le code, affiche seulement le résultat :

```{python}
#| echo: false
print("Code caché, résultat visible")
```

Cette cellule affiche le code, cache le résultat :

```{python}
#| output: false
print("Code visible, résultat caché")
```

Cette cellule est entièrement exclue du document rendu :

```{python}
#| include: false
print("Invisible dans le document final")
```
"""

with open("ex1_cell_options.qmd", "w", encoding="utf-8") as f:
    f.write(qmd)

print("ex1_cell_options.qmd créé.")

In [ ]:
!quarto render ex1_cell_options.qmd

---

### Exemple 2 — Figures avec caption et label

Quarto peut capturer automatiquement les graphiques matplotlib, leur assigner un titre
(`fig-cap`) et un label (`label`) pour les référencer dans le texte avec `@fig-xxx`.

C'est l'équivalent des `\label{}` et `\ref{}` en LaTeX — mais en Markdown.

In [12]:
qmd = """\
---
title: "Exemple 2 — Figures"
format: html
jupyter: ai_learning
---

Le graphique @fig-sinus montre une période complète d'un sinus.

```{python}
#| label: fig-sinus
#| fig-cap: "Une période complète de sin(x)"
#| echo: false

import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 2 * np.pi, 300)
plt.plot(x, np.sin(x))
plt.xlabel("x (radians)")
plt.ylabel("sin(x)")
plt.grid(True)
plt.tight_layout()
plt.show()
```

La référence `@fig-sinus` dans le texte ci-dessus est résolue automatiquement
par Quarto en "Figure 1".
"""

with open("ex2_figures.qmd", "w", encoding="utf-8") as f:
    f.write(qmd)

print("ex2_figures.qmd créé.")

ex2_figures.qmd créé.


In [13]:
!quarto render ex2_figures.qmd


Starting ai_learning kernel...Done

Executing 'ex2_figures.quarto_ipynb'
  Cell 1/1: 'fig-sinus'...Done

pandoc 
  to: html
  output-file: ex2_figures.html
  standalone: true
  section-divs: true
  html-math-method: mathjax
  wrap: none
  default-image-extension: png
  variables: {}
  
metadata
  document-css: false
  link-citations: true
  date-format: long
  lang: en
  engines:
    - path: C:\Program Files\Quarto\share\extension-subtrees\julia-engine\_extensions\julia-engine\julia-engine.js
  title: Exemple 2 — Figures
  jupyter: ai_learning
  
Output created: ex2_figures.html



In [ ]:
!start ex3_params.html

In [14]:
! quarto render ex3_params.qmd -P dataset:mnist -P n_samples:1000


Executing 'ex3_params.quarto_ipynb'
  Cell 1/3: ''...Done
  Cell 2/3: ''...Done
  Cell 3/3: ''...Done

pandoc 
  to: html
  output-file: ex3_params.html
  standalone: true
  section-divs: true
  html-math-method: mathjax
  wrap: none
  default-image-extension: png
  variables: {}
  
metadata
  document-css: false
  link-citations: true
  date-format: long
  lang: en
  engines:
    - path: C:\Program Files\Quarto\share\extension-subtrees\julia-engine\_extensions\julia-engine\julia-engine.js
  title: Exemple 3 — Rapport paramétré
  jupyter: ai_learning
  
Output created: ex3_params.html



In [15]:
!start ex3_params.html

---

### Exemple 3 — Paramètres et rapport reproductible

Le YAML front matter peut contenir des **paramètres** (`params`) accessibles depuis
le code Python via `quarto.params`. Cela permet de générer plusieurs versions du même
rapport en changeant uniquement les paramètres — sans toucher au code.

Cas d'usage typique en AI : générer un rapport d'évaluation de modèle pour différents
datasets ou hyperparamètres.

In [ ]:
qmd = """\
---
title: "Exemple 3 — Rapport paramétré"
format: html
jupyter: ai_learning
---

```{python}
#| include: false
#| tags: [parameters]
dataset = "iris"
n_samples = 50
```

```{python}
#| echo: false
import numpy as np

rng = np.random.default_rng(42)
data = rng.normal(loc=0, scale=1, size=n_samples)

print(f"Dataset   : {dataset}")
print(f"N         : {n_samples}")
print(f"Moyenne   : {data.mean():.4f}")
print(f"Écart-type: {data.std():.4f}")
```

Pour générer ce rapport avec des paramètres différents depuis le terminal :

```bash
quarto render ex3_params.qmd -P dataset:mnist -P n_samples:1000
```
"""

with open("ex3_params.qmd", "w", encoding="utf-8") as f:
    f.write(qmd)

print("ex3_params.qmd créé.")

In [ ]:
!quarto render ex3_params.qmd

---

### Nettoyage des fichiers générés

In [ ]:
import os, shutil

files = [
    "ex1_cell_options.qmd", "ex1_cell_options.html",
    "ex2_figures.qmd",      "ex2_figures.html",
    "ex3_params.qmd",       "ex3_params.html",
]
dirs = ["ex1_cell_options_files", "ex2_figures_files", "ex3_params_files"]

for f in files:
    if os.path.exists(f):
        os.remove(f)
        print(f"Supprimé : {f}")

for d in dirs:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Supprimé : {d}/")